In [18]:
%pip install python-constraint

Note: you may need to restart the kernel to use updated packages.


In [19]:
from constraint import Problem # the constraint package
from constraint import AllDifferentConstraint
from itertools import product # cartesian product

## Setup

In [20]:
problem = Problem() # this is the CSP problem in which the Sudoku should be modeled

# construct list of coordinates of all board cells
# this can be used for the next parts, but you can also use your own approach
board = list(product(range(9), range(9)))
print(board)

[(0, 0), (0, 1), (0, 2), (0, 3), (0, 4), (0, 5), (0, 6), (0, 7), (0, 8), (1, 0), (1, 1), (1, 2), (1, 3), (1, 4), (1, 5), (1, 6), (1, 7), (1, 8), (2, 0), (2, 1), (2, 2), (2, 3), (2, 4), (2, 5), (2, 6), (2, 7), (2, 8), (3, 0), (3, 1), (3, 2), (3, 3), (3, 4), (3, 5), (3, 6), (3, 7), (3, 8), (4, 0), (4, 1), (4, 2), (4, 3), (4, 4), (4, 5), (4, 6), (4, 7), (4, 8), (5, 0), (5, 1), (5, 2), (5, 3), (5, 4), (5, 5), (5, 6), (5, 7), (5, 8), (6, 0), (6, 1), (6, 2), (6, 3), (6, 4), (6, 5), (6, 6), (6, 7), (6, 8), (7, 0), (7, 1), (7, 2), (7, 3), (7, 4), (7, 5), (7, 6), (7, 7), (7, 8), (8, 0), (8, 1), (8, 2), (8, 3), (8, 4), (8, 5), (8, 6), (8, 7), (8, 8)]


## CSP Variables

In [21]:
# TODO create the CSP Variables here using problem.addVariable or add_Variables

# Create a for loop to create variable to all cells and assign their domain
# (i, j) is the cell coordinate: (column, row)

for i in range (9):
    for j in range (9):
        problem.addVariable((i, j), [1, 2, 3, 4, 5, 6, 7, 8, 9])

variables = problem._variables

# print(variables)

## CSP Constraints

Basic Sukoku constraint: Numbers in each 3x3 block cannot be repeated

In [22]:
# TODO your constraints should go here, using problem.addConstraint

for start_row in [0, 3, 6]:
    for start_col in [0, 3, 6]:

        block = []

        # Collect all cells in the current 3x3 block
        for j in range(start_row, start_row + 3):
            for i in range(start_col, start_col + 3):
                block.append((i, j))

        # Numbers in the block must be different
        problem.addConstraint(AllDifferentConstraint(), block)


Basic Sukoku constraint: Numbers in each row cannot be repeated

In [23]:
# TODO your constraints should go here

for j in range(9):
    problem.addConstraint(AllDifferentConstraint(), [(i, j) for i in range(9)])

Basic Sukoku constraint: Numbers in each column cannot be repeated

In [24]:
# TODO your constraints should go here

for i in range(9):
    problem.addConstraint(AllDifferentConstraint(), [(i, j) for j in range(9)])

Some cells are already filled in (see board in exercise)

In [25]:
# TODO your constraints should go here

# Cells whose values are already specified (x, y) = (cols, rows)
given = {
    (4, 1): 2,
    (4, 2): 7,
    (1, 4): 1,
    (2, 4): 6,
    (6, 4): 8,
    (7, 4): 3,
    (4, 6): 5,
    (4, 7): 4
}

# Add constraints for those cells to be assigned their respective values
for cell, value in given.items():
    problem.addConstraint(lambda x, value: x == value, [cell])

Constraint 2: the corners should add up to 10

In [26]:
# TODO your constraints should go here

problem.addConstraint(lambda a, b, c, d: a + b + c + d == 10, [(0, 0), (0, 8), (8, 0), (8, 8)])

Constraint 3: the corners of the bottom left square should add to twice the center

In [27]:
# TODO your constraints should go here

problem.addConstraint(lambda a, b, c, d, e: a + b + c + d == 2 * e, [(0, 6), (0, 8), (2, 8), (2, 6), (1, 7)])

Constraint 4: the cells of the top left square should satisfy certain equations

In [28]:
# TODO your constraints should go here

problem.addConstraint(lambda a, b: a + b == 11, [(0, 0), (1, 0)])
problem.addConstraint(lambda c, d: c - d == 6, [(2, 0), (2, 1)])
problem.addConstraint(lambda e, f: e == 2 * f, [(0, 1), (1, 1)])
problem.addConstraint(lambda g, h, i: g * h * i == 90, [(0, 2), (1, 2), (2, 2)])

Constraint 5: the cellls of the bottom right square should be ordered from top to bottom

In [29]:
# TODO your constraints should go here

problem.addConstraint(lambda a, b, c: a < b < c, [(6, 6), (6, 7), (6, 8)])
problem.addConstraint(lambda d, e, f: d < e < f, [(7, 6), (7, 7), (7, 8)])
problem.addConstraint(lambda g, h, i: g < h < i, [(8, 6), (8, 7), (8, 8)])

Constraint 6: the corners of the top right square should have an adjacent cell that is one larger than them

In [30]:
# TODO your constraints should go here

problem.addConstraint(lambda a, adj1, adj2: adj1 == a + 1 or adj2 == a + 1, [(6, 0), (7, 0), (6, 1)])
problem.addConstraint(lambda b, adj3, adj4: adj3 == b + 1 or adj4 == b + 1, [(8, 0), (7, 0), (8, 1)])
problem.addConstraint(lambda c, adj5, adj6: adj5 == c + 1 or adj6 == c + 1, [(6, 2), (7, 2), (6, 1)])
problem.addConstraint(lambda d, adj7, adj8: adj7 == d + 1 or adj8 == d + 1, [(8, 2), (7, 2), (8, 1)])

Constraint 7: Exactly one of A or B in the center square should be even and the other one odd

In [31]:
# TODO your constraints should go here

problem.addConstraint(
    lambda a, b1, b2, b3, b4: all(b % 2 != a % 2 for b in (b1, b2, b3, b4)),
    [(4, 4), (4, 3), (3, 4), (5, 4), (4, 5)]
)

## Compute and Print Solution

In [32]:
def print_solution(sol):
    board_str = ""
    for y in range(9):
        if y % 3 == 0 and y > 0:
            board_str += "---------------------\n"
        for x in range(9):
            if x % 3 == 0 and x > 0:
                board_str += "| "
            board_str += f"{sol[(x, y)]} " # TODO adjust this to your approach of specifying the CSP variables
        board_str += "\n"
        
    print(board_str)

In [33]:
solutions = problem.getSolutions()

TypeError: <lambda>() missing 1 required positional argument: 'value'

In [ ]:
print(f"Number of solutions: {len(solutions)}\n")
print_solution(solutions[0])

Number of solutions: 0



IndexError: list index out of range